In [ ]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
#from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import yaml
import os
import logging
from contextlib import nullcontext
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

mlflow_tracking_uri = 'http://localhost:5555'  # Optional: e.g., 'http://localhost:5555'

In [ ]:
# Load dataset
train_df = pd.read_csv('train_final.csv')
test_df = pd.read_csv('test_final.csv')

In [ ]:
# 6 tuần cuối: Tuần 26, 27, 28, 29, 30, 31 của năm 2015
val_condition = (train_df['Year'] == 2015) & (train_df['WeekOfYear'] >= 26)

train_set = train_df[~val_condition].copy()
val_set = train_df[val_condition].copy()

In [ ]:
# Thêm vào sau phần Feature Engineering của bạn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# Định nghĩa các model muốn thử nghiệm
models = {
    'RidgeRegression': Ridge(),
    'XGBoost': xgb.XGBRegressor(objective='reg:squarederror', tree_method='hist')
    
}

# Định nghĩa lưới tham số (Grid) để GridSearchCV tìm kiếm
# Lưu ý: Với Rossmann, đừng để grid quá rộng vì dữ liệu rất lớn
model_grids = {
    'RidgeRegression': {
        'alpha': [0.1, 1.0, 10.0]
    },
    'XGBoost': {
        'max_depth': [9, 11],
        'learning_rate': [0.02, 0.09],
        'n_estimators': [100, 500] 
    }
}

In [ ]:
# Tạo hàm để tính RMSPE
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true)**2))

In [ ]:
from sklearn.model_selection import GridSearchCV

def evaluate_model_rossmann(name, model, grid, X_train, y_train, X_val, y_val):
    if grid:
        # Sử dụng GridSearchCV để tìm tham số tốt nhất
        clf = GridSearchCV(model, grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
        clf.fit(X_train, y_train)
        best_model = clf.best_estimator_
        best_params = clf.best_params_
    else:
        model.fit(X_train, y_train)
        best_model = model
        best_params = model.get_params()

    # Dự đoán (nhớ dùng np.exp vì y_train/y_val đang ở dạng log)
    y_pred_log = best_model.predict(X_val)
    y_val_orig = np.exp(y_val)
    y_pred_orig = np.exp(y_pred_log)

    # Tính RMSPE (Metric chính của Rossmann)
    val_rmspe = rmspe(y_val_orig, y_pred_orig)
    
    return {
        'rmspe': val_rmspe,
        'model': best_model,
        'params': best_params
    }

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Rossmann_Experiment_V2")

results = {}

with mlflow.start_run(run_name="Model_Comparison_Suite"):
    for name, model in models.items():
        print(f"Đang thực hiện thí nghiệm cho: {name}...")
        
        with mlflow.start_run(run_name=name, nested=True):
            # 1. Chạy đánh giá
            evaluation = evaluate_model_rossmann(name, model, model_grids[name], X_train, y_train, X_val, y_val)
            results[name] = evaluation

            # 2. Log các thông số lên MLflow
            mlflow.log_params(evaluation['params'])
            mlflow.log_metric("val_rmspe", evaluation['rmspe'])
            
            # 3. Lưu model artifact
            mlflow.sklearn.log_model(evaluation['model'], artifact_path=name.lower())
            
            print(f"✅ {name} - RMSPE: {evaluation['rmspe']:.4f}")

# Vẽ biểu đồ so sánh sau khi chạy xong
import matplotlib.pyplot as plt

names = list(results.keys())
values = [results[name]['rmspe'] for name in names]

plt.figure(figsize=(10, 5))
plt.bar(names, values, color='skyblue')
plt.title('So sánh RMSPE giữa các Model (Thấp hơn là tốt hơn)')
plt.ylabel('RMSPE')
plt.show()

In [ ]:
import yaml
import os

# Tìm model tốt nhất (RMSPE thấp nhất)
best_model_name = min(results, key=lambda x: results[x]['rmspe'])
best_info = results[best_model_name]

model_config = {
    'project': 'Rossmann Store Sales',
    'best_model': {
        'name': best_model_name,
        'rmspe': float(best_info['rmspe']),
        'params': best_info['params']
    },
    'features': {
        'input_columns': list(X_train.columns),
        'target': 'Sales_log'
    }
}

# Lưu vào thư mục configs (tạo folder nếu chưa có)
config_path = '../configs/model_config.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, 'w') as f:
    yaml.dump(model_config, f, default_flow_style=False)

print(f"Đã lưu cấu hình model tốt nhất tại: {config_path}")